# 2. Fit an interpretable detector and compare it fairly

Per device: subtract its normal received power, divide by its robust spread,
and smooth the resulting deterioration score. Higher means sustained loss.
The larger of the downstream/upstream scores triggers a warning after two readings.
Short missing periods mean unknown, not recovered. Two low readings confirm recovery.

Fit references on the first 40%, calibrate on the next 30%, evaluate on the next 15%.
The final 15% is reserved. These fractions and settings are choices, not guarantees.
Isolation Forest is a challenger; the fixed −27 dBm rule is an illustrative baseline,
not a universal receiver specification. All three use the same warning logic.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from telco_anomaly.synthetic import load_config
from telco_anomaly.experiment import split_times, load_data

CONFIG, relative_path = load_config(ROOT / "configs/synthetic.yml")
DATA = ROOT / relative_path
RUN = ROOT / "outputs/simple_model"


In [ ]:
from telco_anomaly.experiment import benchmark

SMOOTHING_HOURS = 1  # Longer smoothing rejects brief noise but delays detection.
SENSITIVITY = 6      # Higher values demand stronger evidence and can miss faults.

if not RUN.exists():
    benchmark(DATA, RUN, SMOOTHING_HOURS, SENSITIVITY)
settings = json.loads((RUN / "settings.json").read_text())
assert settings["smoothing_hours"] == SMOOTHING_HOURS
assert settings["sensitivity"] == SENSITIVITY
assert Path(settings["dataset"]) == DATA.resolve()
# Use a new RUN folder for a new experiment; old results are never overwritten.
comparison = pd.read_csv(RUN / "comparison.csv")
columns = ["model", "faults", "detected", "missed", "false_alarms",
           "duplicate_warnings", "incidents", "impact_faults", "early_warnings",
           "median_positive_lead_hours", "score_coverage"]
display(comparison[columns])


A detection is not necessarily early. `early_warnings / impact_faults` measures
warnings before simulated service impact, including misses in the denominator.
Positive lead time describes successes only. Duplicate warnings include several
ONTs affected by one shared fault and repeated warnings within a fault.
Always inspect total workload as well as false alarms. Thresholds are not matched
to identical workloads, and neither benchmark has been exhaustively tuned.

In [ ]:
model = json.loads((RUN / "model.json").read_text())
scores = pd.read_parquet(RUN / "scores.parquet")
entity = scores.ont_id.iloc[0]
view = scores.loc[scores.ont_id.eq(entity)].set_index("timestamp_utc")
ax = view.score.plot(figsize=(12, 4), ylabel="Smoothed standardised drop",
                     title=f"Development score: {entity}")
ax.axhline(model["threshold"], color="red", label="Warning threshold")
ax.axhline(model["recovery_threshold"], color="green", label="Recovery threshold")
ax.legend()
plt.show()
